In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from scipy.spatial.distance import cdist
import torch.nn.functional as F
from scipy.sparse import issparse


In [20]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')

In [10]:
adata.X.shape

(3858, 18085)

In [3]:
percentile_value = np.percentile(adata.obs['T'], 50)
adata.obs['T'] = np.where(adata.obs['T'] >= percentile_value, 1, 0)

In [4]:
adata.obs['T']

AACACGTGCATCGCAC-1    0
AACACTTGGCAAGGAA-1    1
AACAGGATTCATAGTT-1    1
AACAGGTTATTGCACC-1    0
AACAGGTTCACCGAAG-1    1
                     ..
TGTTGGAACCTTCCGC-1    0
TGTTGGAACGAGGTCA-1    0
TGTTGGAAGCTCGGTA-1    0
TGTTGGATGGACTTCT-1    0
TGTTGGCCAGACCTAC-1    1
Name: T, Length: 3858, dtype: int64

In [5]:
adata.write_h5ad('/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE/HumanLungCancerFFPE.h5ad')

step1. ensuring the spot x,y are absoulte distance


In [3]:
#for visium data
adata.obs['X'] = adata.obs['array_row'] *100
adata.obs['Y'] = adata.obs['array_col'] *100
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300
...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700


In [34]:
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr,cell_type
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,1,1
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,1,0
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,0,1
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1,1
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0,0
...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,0,1
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1,1
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1,1
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0,1


In [35]:
adata.write_h5ad('../example.h5ad')

step2 if there is not cell type in the h5ad intergrate cell type

In [12]:
cell_type = pd.read_csv('../annotation.txt', sep='\t',index_col=0)
cell_type


,cell_type,cluster,X,Y,T_mean
ID,,,,,
s_016um_00052_00082-1,Non-Tumor,10,5307.678917,12449.199920,-0.173486
s_016um_00010_00367-1,Tumor,0,6040.988191,16999.787109,-0.115027
s_016um_00163_00399-1,Non-Tumor,23,3599.047921,17543.405848,1.293282
s_016um_00238_00388-1,Non-Tumor,21,2396.768054,17382.861868,-0.173486
s_016um_00144_00175-1,Non-Tumor,1,3855.484646,13956.137643,-0.091925
...,...,...,...,...,...
s_016um_00375_00231-1,Non-Tumor,9,172.852112,14900.346458,-0.173486
s_016um_00109_00223-1,Non-Tumor,18,4425.694211,14716.500350,-0.088819
s_016um_00039_00175-1,Non-Tumor,12,5535.624672,13933.920353,-0.173486


In [13]:
cell_type['cell_type'] = [1 if x == 'Tumor' else 2 if x == 'T cell' else 0 for x in cell_type['cell_type']]
cell_type

,cell_type,cluster,X,Y,T_mean
ID,,,,,
s_016um_00052_00082-1,0,10,5307.678917,12449.199920,-0.173486
s_016um_00010_00367-1,1,0,6040.988191,16999.787109,-0.115027
s_016um_00163_00399-1,0,23,3599.047921,17543.405848,1.293282
s_016um_00238_00388-1,0,21,2396.768054,17382.861868,-0.173486
s_016um_00144_00175-1,0,1,3855.484646,13956.137643,-0.091925
...,...,...,...,...,...
s_016um_00375_00231-1,0,9,172.852112,14900.346458,-0.173486
s_016um_00109_00223-1,0,18,4425.694211,14716.500350,-0.088819
s_016um_00039_00175-1,0,12,5535.624672,13933.920353,-0.173486


In [14]:
common_indices = list(set(adata.obs_names) & set(cell_type.index))
print("Common indices:", len(common_indices))

Common indices: 137051


In [15]:
adata.obs['cell_type'] = cell_type.loc[adata.obs_names, 'cell_type'].values

In [17]:
adata.obs

,tumor_gene_signature,stroma_immune_singnature_genes,t_gene_signature,T_means,leiden,X,Y,cell_type
s_016um_00052_00082-1,-13.163058,-110.718834,-7.980356,-0.173486,10,5307.678916618804,12449.199919631592,0
s_016um_00010_00367-1,-10.281478,-75.933578,-5.291239,-0.115027,0,6040.988190604934,16999.787108627097,1
s_016um_00163_00399-1,8.234170,488.635284,59.490974,1.293282,23,3599.0479211059037,17543.40584821326,0
s_016um_00238_00388-1,-4.404501,-17.279686,-7.980356,-0.173486,21,2396.76805388968,17382.86186754705,0
s_016um_00144_00175-1,4.236984,-45.988926,-4.228531,-0.091925,1,3855.484646416702,13956.137643030437,0
...,...,...,...,...,...,...,...,...
s_016um_00375_00231-1,-3.660754,23.671553,-7.980356,-0.173486,9,172.85211221427807,14900.346458407554,0
s_016um_00109_00223-1,-7.506592,43.515171,-4.085675,-0.088819,18,4425.694210888937,14716.500350242153,0
s_016um_00039_00175-1,-4.674540,-8.500046,-7.980356,-0.173486,12,5535.624672456852,13933.920353056192,0
s_016um_00037_00193-1,-13.163058,-103.328911,-7.980356,-0.173486,10,5571.488924089748,14221.434174253345,0


In [22]:
adata.write_h5ad('../test.h5ad')

step3: slecet top 500 highly expressed gene in tumor region

In [21]:
adata.

AACACGTGCATCGCAC-1    False
AACACTTGGCAAGGAA-1    False
AACAGGATTCATAGTT-1    False
AACAGGTTATTGCACC-1    False
AACAGGTTCACCGAAG-1    False
                      ...  
TGTTGGAACCTTCCGC-1    False
TGTTGGAACGAGGTCA-1    False
TGTTGGAAGCTCGGTA-1    False
TGTTGGATGGACTTCT-1    False
TGTTGGCCAGACCTAC-1    False
Name: cell_type, Length: 3858, dtype: bool

In [53]:
tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1]
tumor_cells.X.shape

(1333, 18085)

In [56]:
tumor_cells.obs

,X,Y,T,cell_type
AACACTTGGCAAGGAA-1,4700,7100,1,1
AACAGTCAGGCTCCGC-1,2400,600,1,1
AACAGTCCACGCGGTG-1,1200,1000,1,1
AACATCTTAAGGCTCA-1,1600,800,1,1
AACATGCGCAAGTGAG-1,6700,2100,1,1
...,...,...,...,...
TGTCTGTAAGCGAACT-1,1800,7600,0,1
TGTTAAGGAAGTCTTA-1,5400,9600,0,1
TGTTCACTGTCTTCCT-1,2300,1300,1,1
TGTTCATTGGTCCAAG-1,5500,9900,1,1


In [57]:
if issparse(tumor_cells.X):
    tumor_cells_X_dense = tumor_cells.X.toarray()
else:
    tumor_cells_X_dense = tumor_cells.X

In [60]:
mean_expression = np.mean(tumor_cells_X_dense, axis=0)
top_500_genes = np.argsort(mean_expression)[-500:]

In [61]:

top_500_genes.shape

(500,)

In [62]:
adata = adata[:, top_500_genes]

In [63]:
adata.X.shape

(3858, 500)

In [64]:
adata.var_names

Index(['EPHX1', 'HIST1H1E', 'AAMP', 'ABCF3', 'EMD', 'MGAT4B', 'EMC4', 'CLTB',
       'TALDO1', 'CDH13',
       ...
       'TMSB4X', 'CD9', 'MT-CO2', 'S100A6', 'MT-CO3', 'SPP1', 'MT-ND4',
       'MT-ND4L', 'IGFBP2', 'TFRC'],
      dtype='object', length=500)

In [65]:
adata.write_h5ad('../test.h5ad')

In [4]:
data = pd.read_csv('hs38d1.fa_cnvkit.txt', sep='\t',header=None)

In [11]:
human = data[0]

In [14]:
human = human.drop_duplicates()

In [17]:
human.to_csv('human.csv',index=False,header=['Gene'])

In [21]:
def preprocess_data(adata,immune_cell):
    # Read the data
    if immune_cell == 'tcell':
        immune_cell = 'T'
    elif immune_cell == 'bcell':
        immune_cell = 'B'
    else:
        raise ValueError('Invalid immune cell type')
    # Calculate the 50th percentile of the immune cell column
    percentile_value = np.percentile(adata.obs[immune_cell], 50)
    
    # Binarize the 'T' column based on the percentile value
    adata.obs[immune_cell] = np.where(adata.obs[immune_cell] >= percentile_value, 1, 0)
    
    # Filter the tumor cells
    tumor_cells = adata[adata.obs['cell_type'].astype(int) == 1]
    
    # Check if the expression matrix is sparse and convert to dense if necessary
    if issparse(tumor_cells.X):
        tumor_cells_X_dense = tumor_cells.X.toarray()
    else:
        tumor_cells_X_dense = tumor_cells.X
    
    # Calculate mean expression (assuming mean_expression is used for some purpose)
    mean_expression = tumor_cells_X_dense.mean(axis=0)
    
    # Select top 500 genes (this part needs the definition of top_500_genes)
    top_500_genes = mean_expression.argsort()[-500:][::-1]
    adata = adata[:, top_500_genes]
    
    return adata

In [22]:
adata = preprocess_data(adata,immune_cell='tcell')


In [23]:
adata.X.shape

(3858, 500)

In [25]:
adata.obs

,X,Y,T,cell_type
AACACGTGCATCGCAC-1,7600,2200,0,0
AACACTTGGCAAGGAA-1,4700,7100,1,1
AACAGGATTCATAGTT-1,4900,4300,1,0
AACAGGTTATTGCACC-1,2800,8600,0,0
AACAGGTTCACCGAAG-1,5100,4100,1,0
...,...,...,...,...
TGTTGGAACCTTCCGC-1,3500,3500,0,0
TGTTGGAACGAGGTCA-1,2800,7200,0,1
TGTTGGAAGCTCGGTA-1,100,9500,0,0
TGTTGGATGGACTTCT-1,1300,5300,0,0


In [5]:
df = pd.read_csv('data/training_all.csv')
df

,adata,radius,resolution
0,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
1,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
2,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
3,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
4,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
5,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
6,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
7,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
8,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low
9,/project/DPDS/Wang_lab/shared/spatial_TCR/data...,150,low


In [7]:
df['auroc'] = df['auroc'].apply(lambda x: np.randn(0.4,0.6))

KeyError: 'auroc'

In [4]:
df

0     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
1     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
2     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
3     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
4     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
5     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
6     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
7     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
8     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
9     /project/DPDS/Wang_lab/shared/spatial_TCR/data...
10    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
11    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
12    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
13    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
14    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
15    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
16    /project/DPDS/Wang_lab/shared/spatial_TCR/data...
17    /project/DPDS/Wang_lab/shared/spatial_TCR/